###Run Shared Libraries

In [0]:
%run ../Misc/sharedlibraries

In [0]:
UpdatedDateTime = datetime.datetime.now()
Entity = "factpurchaseorder"

In [0]:
purchaseorderDf = spark.table("bronze.purchaseorder")
dimcostcenterDf = spark.table("silver.dimcostcenter")
dimcurrencyDf = spark.table("silver.dimcurrency")

In [0]:
purchaseorderDf.printSchema()
dimcostcenterDf.printSchema()
dimcurrencyDf.printSchema()

###Build Fact Purchase Order table

In [0]:
po = purchaseorderDf.alias("po")
cc = dimcostcenterDf.alias("cc")
cur = dimcurrencyDf.alias("cur")

factpurchaseorderDf = po.filter(
        F.col("po.RecordId").isNotNull()
    ).join(
        cc,
        F.col("po.CostCenter") == F.col("cc.CostCenterNumber"),
        "left"
    ).join(
        cur,
        F.col("po.CurrencyCode") == F.col("cur.CurrencyCode"),
        "left"
    ).select(
        F.col("po.PoNumber"),
        F.col("po.LineItem"),
        F.col("po.VendId").alias("VendorKey"),

        F.when(
            F.col("po.LastProcessedChange_DateTime").isNull(),
            F.lit("1900-01-01")
        ).otherwise(
            F.col("po.LastProcessedChange_DateTime")
        ).cast("timestamp").alias("LastProcessedChange_DateTime"),

        F.from_utc_timestamp(F.col("po.DataLakeModified_DateTime"), "CST").alias("DataLakeModified_DateTime"),

        F.col("po.Qty"),
        F.col("po.PurchasePrice"),
        F.col("po.TotalOrder"),

        F.col("po.CostCenter").alias("CostCenterKey"),
        F.col("cc.Vat").alias("VatAmount"),

        F.round(
            F.col("po.TotalOrder") + (F.col("po.TotalOrder") * F.col("cc.Vat")),
            4
        ).alias("TotalAmount"),

        F.col("po.ExchangeRate"),
        F.col("po.ItemKey"),
        F.col("cur.CurrencyId").alias("CurrencyKey"),

        F.from_utc_timestamp(F.col("po.OrderDate"), "CST").alias("OrderDate"),
        F.from_utc_timestamp(F.col("po.ShipDate"), "CST").alias("ShipDate"),
        F.from_utc_timestamp(F.col("po.DeliveredDate"), "CST").alias("DeliveredDate"),

        F.date_format(F.col("po.OrderDate"), "yyyyMMdd").cast("int").alias("OrderDateKey"),
        F.date_format(F.col("po.ShipDate"), "yyyyMMdd").cast("int").alias("ShipDateKey"),
        F.date_format(F.col("po.DeliveredDate"), "yyyyMMdd").cast("int").alias("DeliveredDateKey"),

        F.col("po.TrackingNumber"),
        F.col("po.BatchId"),
        F.col("po.CreatedBy"),
        F.col("po.RecordId").alias("PurchaseOrderRecordId"),
        F.col("po.CategoryId").alias("CategoryKey")
    ).withColumn(
        "UpdatedDateTime",
        F.lit(UpdatedDateTime)
    ).withColumn(
        "PurchaseOrderHashKey",
        F.xxhash64("PurchaseOrderRecordId")
    )

display(factpurchaseorderDf)

###Final dataframe

In [0]:
df_final = factpurchaseorderDf

## Write to Silver Schema

In [0]:
saveDeltaTableToCatalog(df_final,"silver",Entity)

In [0]:
%sql
SHOW TABLES IN bronze;
SHOW TABLES IN silver;